# Pawn counting

Project to explore/visualise chess pawn structures

In [1]:
from typing import Callable

from pathlib import Path
import functools
import numpy as np
import polars as pl
import altair as alt
import scipy.sparse as sp
from sklearn.manifold import spectral_embedding

In [2]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## basic functions

In [3]:
type Position = tuple[np.uint64, np.uint64]
"""
Pawn structure position --- a configuration of White and Black pawns on a board of size
BOARD_WIDTH * BOARD_DEPTH

Consists of two u64 bitmasks, the first for White pawns and the second for Black.

The least bit of each bitmask is the bottom-left square for White (a1), counting up in
file-major order (a1, a2, ..., a8, b1, ...) to the top-right square for White (h8).

If the board is smaller than 8x8, the board crops around bottom-left square for White
(so the bottom-left square remains a1), and the unused squares are empty.
"""

BOARD_WIDTH = 3
BOARD_DEPTH = 4

assert 0 <= BOARD_WIDTH <= 8
assert 0 <= BOARD_DEPTH <= 8

In [4]:
def position_to_int(pos: Position) -> int:
    """Pack a Position into a single UInt128: low 64 bits White, high 64 bits Black."""
    white, black = pos
    return int(white) | (int(black) << 64)

def position_from_int(code: int) -> Position:
    """Unpack a UInt128 (low bits White, high bits Black) back into a Position."""
    mask = (1 << 64) - 1
    return np.uint64(code & mask), np.uint64((code >> 64) & mask)

In [5]:
_CACHE = f"./transitions_{BOARD_WIDTH}x{BOARD_DEPTH}.tmp"


def cached_frame(name: str):
    def cached_frame_inner(func: Callable[..., pl.DataFrame]):
        nonlocal name

        @functools.wraps(func)
        def inner_func(*args, **kwargs):
            Path(_CACHE).mkdir(parents=True, exist_ok=True)

            cached_frame_path = Path(_CACHE) / f"{name}.parquet"
            if cached_frame_path.exists():
                return pl.read_parquet(cached_frame_path)

            frame = func(*args, **kwargs)
            frame.write_parquet(cached_frame_path)
            return frame

        return inner_func

    return cached_frame_inner


In [6]:
def position_ndarray(pos: Position) -> tuple[np.ndarray, np.ndarray]:
    def to_array(mask):
        arr = np.zeros((BOARD_WIDTH, BOARD_DEPTH), dtype=bool)
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                arr[f, r] = (mask >> (f * 8 + r)) & 1
        return arr

    white, black = pos
    return to_array(white), to_array(black)

def init_position() -> Position:
    white = 0
    black = 0
    for f in range(BOARD_WIDTH):
        white |= 1 << (f * 8)
        black |= 1 << (f * 8 + BOARD_DEPTH - 1)
    return np.uint64(white), np.uint64(black)

def rand_position(max_pawns_per_side: int | None = None) -> Position:
    squares = [f * 8 + r for f in range(BOARD_WIDTH) for r in range(BOARD_DEPTH)]
    n = len(squares)
    cap = n if max_pawns_per_side is None else min(max_pawns_per_side, n)

    n_white = np.random.randint(cap + 1)
    n_black = np.random.randint(min(cap, n - n_white) + 1)

    chosen = np.random.choice(n, size=n_white + n_black, replace=False)
    white = 0
    black = 0
    for i in chosen[:n_white]:
        white |= 1 << squares[i]
    for i in chosen[n_white:]:
        black |= 1 << squares[i]
    return np.uint64(white), np.uint64(black)

def pawns_as_frame(pos: Position) -> pl.DataFrame:
    colour = pl.Enum(["White", "Black"])
    rows = []
    for name, mask in zip(["White", "Black"], pos):
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                if (mask >> (f * 8 + r)) & 1:
                    rows.append({"rank": r + 1, "file": f + 1, "colour": name})
    return pl.DataFrame(rows, schema={"rank": pl.Int64, "file": pl.Int64, "colour": colour})

In [7]:
def position_chart(pos: Position, *, axis=False, legend=False) -> alt.LayerChart:
    _PAWN_COLOURS = {
        "domain": ["White", "Black"],
        "range": ["white", "black"],
    }
    _SQUARE_COLOURS = {
        "domain": ["light", "dark"],
        "range": ["#f0d9b5", "#b58863"],
    }
    _FILE_DOMAIN = list(range(1, BOARD_WIDTH + 1))
    _RANK_DOMAIN = list(range(1, BOARD_DEPTH + 1))

    def _board_as_frame() -> pl.DataFrame:
        rows = []
        for f in range(1, BOARD_WIDTH + 1):
            for r in range(1, BOARD_DEPTH + 1):
                square = "dark" if (f + r) % 2 == 0 else "light"
                rows.append({"rank": r, "file": f, "square": square})
        return pl.DataFrame(rows)

    axis_params = {} if axis else {"axis": None}
    legend_params = {} if legend else {"legend": None}

    board = (
        alt.Chart(_board_as_frame())
        .mark_rect()
        .encode(
            alt.X("file:O", **axis_params).scale(domain=_FILE_DOMAIN),  # type: ignore
            alt.Y("rank:O", **axis_params).scale(domain=_RANK_DOMAIN),  # type: ignore
            alt.Color("square:N")  # type: ignore
            .scale(**_SQUARE_COLOURS)  # type: ignore
            .legend(None),
        )
    )

    pawns = (
        alt.Chart(pawns_as_frame(pos))
        .mark_circle(size=250, stroke="black", strokeWidth=0.5)
        .encode(
            alt.X("file:O").scale(domain=_FILE_DOMAIN),
            alt.Y("rank:O").scale(domain=_RANK_DOMAIN, reverse=True),
            alt.Color("colour:N", **legend_params)  # type: ignore
            #
            .scale(**_PAWN_COLOURS),  # type: ignore
        )
    )

    return (
        alt.layer(board, pawns)
        .resolve_scale(color="independent")
        .properties(width=33 * BOARD_WIDTH, height=33 * BOARD_DEPTH)
    )  # type: ignore


position_chart(init_position())

alt.LayerChart(...)

In [8]:
def _on_board(f: int, r: int) -> bool:
    return 0 <= f < BOARD_WIDTH and 0 <= r < BOARD_DEPTH

def accessible_positions(pos: Position) -> list[tuple[str, int, Position]]:
    white, black = int(pos[0]), int(pos[1])
    out: list[tuple[str, int, Position]] = []

    def emit(kind, src, new_own, new_enemy, white_to_move):
        w, b = (new_own, new_enemy) if white_to_move else (new_enemy, new_own)
        out.append((kind, src, (np.uint64(w), np.uint64(b))))

    for white_to_move in (True, False):
        own, enemy = (white, black) if white_to_move else (black, white)
        dr = 1 if white_to_move else -1
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                src = f * 8 + r
                if not (own >> src) & 1:
                    continue
                own_removed = own & ~(1 << src)

                af, ar = f, r + dr  # advance (straight)
                if _on_board(af, ar):
                    dst = af * 8 + ar
                    if not ((own | enemy) >> dst) & 1:
                        emit("A", src, own_removed | (1 << dst), enemy, white_to_move)
                else:
                    emit("A", src, own_removed, enemy, white_to_move)  # off end rank == R

                for kind, df in (("CQ", -1), ("CK", 1)):  # diagonal captures
                    cf, cr = f + df, r + dr
                    if not 0 <= cf < BOARD_WIDTH:
                        continue  # off the side: not a legal capture
                    if not 0 <= cr < BOARD_DEPTH:
                        emit(kind, src, own_removed, enemy, white_to_move)  # off end rank == R
                        continue
                    dst = cf * 8 + cr
                    if (own >> dst) & 1:
                        continue  # friendly block
                    emit(
                        kind,
                        src,
                        own_removed | (1 << dst),
                        enemy & ~(1 << dst),
                        white_to_move,
                    )

                emit("R", src, own_removed, enemy, white_to_move)  # always

    return out

In [9]:
def multiple_positions_chart(poses: list[Position], *, width: int | None = None) -> alt.VConcatChart:
    pos_charts = [position_chart(p) for p in poses]
    if width is None:
        width = int(np.sqrt(len(pos_charts)))
    return alt.vconcat(
        *(
            alt.hconcat(*pos_charts[n : (n + width)])
            for n in range(0, len(pos_charts), width)
        )
    )

multiple_positions_chart([p for _, _, p in accessible_positions(init_position())], width=7)

alt.VConcatChart(...)

## generation

In [10]:
def explore_transitions(start: Position) -> pl.DataFrame:
    """Every transition reachable from `start`, one row per transition.

    Positions are packed into a UInt128: low 64 bits White, high 64 bits Black.
    `transition_depth` is the minimum number of steps to reach the position the
    move is made from (the BFS level of `start_pos`).

    Edges are collected one BFS level at a time and concatenated, keeping peak
    memory bounded (~2.4 GB / ~2 min for the 4x4 start position: ~2.2M positions,
    ~46M transitions).
    """
    transition_type = pl.Enum(["A", "CQ", "CK", "R"])
    schema = {
        "start_pos": pl.UInt128,
        "end_pos": pl.UInt128,
        "moving_pawn": pl.UInt8,
        "transition_type": transition_type,
    }

    def encode(pos: Position) -> int:
        return int(pos[0]) | (int(pos[1]) << 64)

    seen = {encode(start)}
    frontier = [start]
    chunks: list[pl.DataFrame] = []

    depth = 0
    while frontier:
        starts, ends, pawns, kinds = [], [], [], []
        next_frontier = []
        for pos in frontier:
            start_code = encode(pos)
            for kind, src, end in accessible_positions(pos):
                end_code = encode(end)
                starts.append(start_code)
                ends.append(end_code)
                pawns.append(src)
                kinds.append(kind)
                if end_code not in seen:
                    seen.add(end_code)
                    next_frontier.append(end)
        if starts:
            chunks.append(
                pl.DataFrame(
                    {
                        "start_pos": starts,
                        "end_pos": ends,
                        "moving_pawn": pawns,
                        "transition_type": kinds,
                    },
                    schema=schema,
                ).with_columns(transition_depth=pl.lit(depth, dtype=pl.UInt8))
            )
        frontier = next_frontier
        depth += 1

    return pl.concat(chunks, rechunk=False)


@cached_frame("transitions")
def _transitions() -> pl.DataFrame:
    return explore_transitions(init_position())


transitions = _transitions()
print(transitions.schema)
transitions

Schema({'start_pos': UInt128, 'end_pos': UInt128, 'moving_pawn': UInt8, 'transition_type': Enum(categories=['A', 'CQ', 'CK', 'R']), 'transition_depth': UInt8})


start_pos,end_pos,moving_pawn,transition_type,transition_depth
u128,u128,u8,enum,u8
9709333062732580235837697,9709333062732580235837698,0,"""A""",0
9709333062732580235837697,9709333062732580235838208,0,"""CK""",0
9709333062732580235837697,9709333062732580235837696,0,"""R""",0
9709333062732580235837697,9709333062732580235837953,8,"""A""",0
9709333062732580235837697,9709333062732580235837443,8,"""CQ""",0
…,…,…,…,…
1213666632841572529997832,1208944266358702884784136,8,"""CK""",18
1213666632841572529997832,1208944266358702884784136,8,"""R""",18
1213666632841572529997832,4740813226943355291656,16,"""A""",18


In [11]:
def extract_positions(transitions: pl.DataFrame) -> pl.DataFrame:
    return (
        pl.concat(
            [
                transitions.lazy().select(pos="start_pos"),
                transitions.lazy().select(pos="end_pos"),
            ]
        )
        .select(pl.col("pos").unique(maintain_order=True))
        .collect()
)

positions = extract_positions(transitions).with_row_index("position_id")
positions

position_id,pos
u32,u128
0,9709333062732580235837697
1,9709333062732580235837698
2,9709333062732580235838208
3,9709333062732580235837696
4,9709333062732580235837953
…,…
42775,1213666632841572529996808
42776,1213666632841572530257928
42777,1213666632841572529995788


## exploration

### transition depth

In [12]:
# branching factor plot

_plot_df = (
    transitions.group_by("transition_depth", "start_pos")
    .agg(transitions_count=pl.len())
    .group_by("transition_depth")
    .agg(
        transitions_count=pl.col("transitions_count").sum(),
        branching_factor=pl.col("transitions_count").mean(),
    )
    .sort("transition_depth")
)

alt.hconcat(
    alt.Chart(_plot_df).mark_line().encode(x="transition_depth", y="transitions_count"),
    alt.Chart(_plot_df).mark_line().encode(x="transition_depth", y="branching_factor"),
)

alt.HConcatChart(...)

In [13]:
# positions leading to empty
multiple_positions_chart(
    [
        position_from_int(long_pos)
        for long_pos in transitions.filter(
            pl.col("end_pos") == 0, pl.col("transition_type") != "R"
        )["start_pos"]
    ],
    width=7,
)

alt.VConcatChart(...)

### spectral mesh visualisation

In [16]:
def positions_adjacency(
    transitions: pl.DataFrame, positions: pl.DataFrame
) -> sp.csr_matrix:
    """Symmetric 0/1 adjacency of the position graph, indexed by `positions` row order.

    Transitions are directed; we symmetrise (undirected) and drop self-loops so the
    result is a simple undirected graph suitable for a spectral embedding.
    """
    nodes = positions
    edges = (
        transitions.select("start_pos", "end_pos")
        .join(nodes.rename({"pos": "start_pos", "position_id": "i"}), on="start_pos")
        .join(nodes.rename({"pos": "end_pos", "position_id": "j"}), on="end_pos")
    )
    i = edges["i"].to_numpy()
    j = edges["j"].to_numpy()
    n = positions.height
    A = sp.coo_matrix((np.ones(len(i)), (i, j)), shape=(n, n)).tocsr()
    A = A.maximum(A.T)  # undirected
    A.setdiag(0)  # no self-loops
    A.eliminate_zeros()
    A.data[:] = 1.0  # unweighted
    return A  # type: ignore


def spectral_embedding_3d(
    transitions: pl.DataFrame, positions: pl.DataFrame, *, eigen_solver: str = "amg"
) -> pl.DataFrame:
    """3D spectral embedding of the position graph: `positions` plus (x, y, z) columns.

    Rows line up with `positions`. Coordinates are the smallest non-trivial eigenvectors
    of the normalised graph Laplacian. `eigen_solver="amg"` (pyamg-preconditioned LOBPCG)
    scales to millions of nodes; "arpack" (shift-invert) does NOT here -- fill-in on this
    expander-like graph makes it blow past minutes even at 3x4. "lobpcg" is fine small.
    """
    A = positions_adjacency(transitions, positions)
    emb = spectral_embedding(
        A,
        n_components=3,
        eigen_solver=eigen_solver,  # type: ignore
        drop_first=True,
        random_state=0,
    )
    return positions.with_columns(
        x=pl.Series(emb[:, 0]), y=pl.Series(emb[:, 1]), z=pl.Series(emb[:, 2])
    )


@cached_frame("spectral_embeddings")
def _spectral_embeddings() -> pl.DataFrame:
    return spectral_embedding_3d(transitions, positions)


spectral_embeddings = _spectral_embeddings()
spectral_embeddings

position_id,pos,x,y,z
u32,u128,f64,f64,f64
0,9709333062732580235837697,0.002898,-0.000005,0.000002
1,9709333062732580235837698,0.002498,-0.000296,0.000359
2,9709333062732580235838208,0.002585,-0.000501,0.000504
3,9709333062732580235837696,0.002515,-0.000442,0.000453
4,9709333062732580235837953,0.002893,-0.000004,0.000002
…,…,…,…,…
42775,1213666632841572529996808,-0.002708,-0.000004,0.000002
42776,1213666632841572530257928,-0.002639,-0.000314,-0.000241
42777,1213666632841572529995788,-0.00264,0.000309,0.000243


In [29]:
transitions

start_pos,end_pos,moving_pawn,transition_type,transition_depth
u128,u128,u8,enum,u8
9709333062732580235837697,9709333062732580235837698,0,"""A""",0
9709333062732580235837697,9709333062732580235838208,0,"""CK""",0
9709333062732580235837697,9709333062732580235837696,0,"""R""",0
9709333062732580235837697,9709333062732580235837953,8,"""A""",0
9709333062732580235837697,9709333062732580235837443,8,"""CQ""",0
…,…,…,…,…
1213666632841572529997832,1208944266358702884784136,8,"""CK""",18
1213666632841572529997832,1208944266358702884784136,8,"""R""",18
1213666632841572529997832,4740813226943355291656,16,"""A""",18


In [47]:
_DEPTH = 3

_TRANSITION_COLOURS = {
    "domain": ["A", "CK", "CQ", "R"],
    "range": ["gold", "yellowgreen", "mediumseagreen", "brown"],
}

_edges_df = (
    transitions.lazy()
    .filter(pl.col("transition_depth") < _DEPTH)
    .join(
        spectral_embeddings.lazy().select(start_pos="pos", x0="x", y0="y", z0="z"),
        on="start_pos",
    )
    .join(
        spectral_embeddings.lazy().select(end_pos="pos", x1="x", y1="y", z1="z"),
        on="end_pos",
    )
    .collect()
)

(
    alt.Chart(_edges_df)
    .mark_rule(strokeWidth=2, opacity=0.2)
    .encode(
        alt.X("x0"),
        alt.X2("x1"),
        alt.Y("y0"),
        alt.Y2("y1"),
        alt.Color("transition_type:N").scale(**_TRANSITION_COLOURS),  # type: ignore
    )
    .properties(width=700, height=700)
)

alt.Chart(...)